In [13]:
import time
from argparse import ArgumentParser
import gymnasium
import numpy as np
import vizdoom as vzd
from vizdoom import gymnasium_wrapper  # noqa

# Change based on the scenerio
DEFAULT_ENV = "VizdoomBasic-v1"

# print function Return a compact summary without printing an entire screen buffer
def array_summary(value, max_values=5):
 
    value = np.asarray(value)
    return f"shape={value.shape}"


def get_gamevariable_names(env):
    """
    Obtain names of game variables exposed by the ViZDoom scenario.

    The values in obs['gamevariables'] correspond to this order.
    """
    try:
        game_variables = env.unwrapped.game.get_available_game_variables()
        return [variable.name for variable in game_variables]
    except (AttributeError, RuntimeError):
        return []


def print_observation_info(obs, obs_space, gamevariable_names=None, indent=0):
    """Print observation-space metadata and current observation values."""
    prefix = " " * indent

    if isinstance(obs_space, gymnasium.spaces.Dict):
        print(f"{prefix}Observation space: Dict")

        for key, space in obs_space.spaces.items():
            print(f"{prefix}  Key: '{key}'")
            print_observation_info(
                obs[key],
                space,
                gamevariable_names=gamevariable_names if key == "gamevariables" else None,
                indent=indent + 4,
            )

    elif isinstance(obs_space, gymnasium.spaces.Box):
        
        print(f"{prefix}Expected shape: {obs_space.shape}")
        print(f"{prefix}Expected dtype: {obs_space.dtype}")
        print(f"{prefix}Current observation: {array_summary(obs)}")

        # Game variables are usually a 1-D float array.
        if gamevariable_names is not None:
            values = np.asarray(obs).ravel()

            print(f"{prefix}Game variables ({len(values)}):")
            for index, value in enumerate(values):
                name = (
                    gamevariable_names[index]
                    if index < len(gamevariable_names)
                    else f"UNKNOWN_VARIABLE_{index}"
                )
                print(f"{prefix}  [{index}] {name} = {value}")

    elif isinstance(obs_space, gymnasium.spaces.Discrete):
        print(f"{prefix}Space: Discrete(n={obs_space.n})")
        print(f"{prefix}Current observation: {obs}")

    else:
        print(f"{prefix}Space: {obs_space}")
        print(f"{prefix}Current observation: {obs}")

if __name__ == "__main__":
    parser = ArgumentParser()
    parser.add_argument(
        "--env",
        type=str,
        default=DEFAULT_ENV,
        help="The VizDoom environment to run",
    )

    args = parser.parse_args(args=[])

    env = gymnasium.make(
        args.env,
        render_mode="human",
        frame_skip=4,
        screen_resolution=vzd.ScreenResolution.RES_320X240,# Other reso are also possible
    )

    print("\n================ ENVIRONMENT INFORMATION ================")
    print(f"Environment:       {args.env}")
    print(f"Action space:      {env.action_space}")
    print(f"Observation space: {env.observation_space}")
    print(f"Frame skip:        {env.unwrapped.frame_skip}")

    termination_thresh = 20 # terminate after N moves
    # no of episodes
    for i in range(10):
        print(f"\n************ EPISODE NO = {i + 1} ************")

        obs, info = env.reset(seed=77)
        gamevariable_names = get_gamevariable_names(env)

        print_observation_info(
            obs,
            env.observation_space,
            gamevariable_names=gamevariable_names,
        )
        print(f"Reset info: {info}")

        attempt_number = 0
        done = False

        while not done:
            # randomly sampled actions
            action = env.action_space.sample()
            obs, reward, terminated, truncated, info = env.step(action)

            attempt_number += 1
            done = terminated or truncated or attempt_number >= termination_thresh

            print(f"\n--- Step {attempt_number} ---")
            print(f"Action: {action}")
            print_observation_info(
                obs,
                env.observation_space,
                gamevariable_names=gamevariable_names,
            )
            print(f"Reward: {reward}")
            print(f"Terminated: {terminated}, Truncated: {truncated}")
            print(f"Info: {info}")

            time.sleep(1.0 / vzd.DEFAULT_TICRATE * env.unwrapped.frame_skip)

    env.close()


================ ENVIRONMENT INFORMATION ================
Environment:       VizdoomBasic-v1
Action space:      Discrete(4)
Observation space: Dict('gamevariables': Box(-3.4028235e+38, 3.4028235e+38, (1,), float32), 'screen': Box(0, 255, (240, 320, 3), uint8))
Frame skip:        4

************ EPISODE NO = 1 ************
Observation space: Dict
  Key: 'gamevariables'
    Expected shape: (1,)
    Expected dtype: float32
    Current observation: shape=(1,)
    Game variables (1):
      [0] AMMO2 = 50.0
  Key: 'screen'
    Expected shape: (240, 320, 3)
    Expected dtype: uint8
    Current observation: shape=(240, 320, 3)
Reset info: {}

--- Step 1 ---
Action: 0
Observation space: Dict
  Key: 'gamevariables'
    Expected shape: (1,)
    Expected dtype: float32
    Current observation: shape=(1,)
    Game variables (1):
      [0] AMMO2 = 50.0
  Key: 'screen'
    Expected shape: (240, 320, 3)
    Expected dtype: uint8
    Current observation: shape=(240, 320, 3)
Reward: -4.0
Terminated: F